# SOG vs Coulomb 指标对比（cumulene）

本 notebook 对比两个实验目录：

- SOG: `/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1`
- Coulomb: `/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp1`

目标：

1. 统计能量/力的 MAE、RMSE，并计算 MSE（`MSE = RMSE^2`）。
2. 给出 SOG 相对 Coulomb 的改进比例。
3. 判断 SOG 核参数是否参与训练（基于配置与参数规模证据）。

In [1]:
from pathlib import Path
import yaml
import pandas as pd

sog_dir = Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-sog-cu30-lr-mp1')
coulomb_dir = Path('/data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/lorem-cu30-lr-mp1')

paths = {
    'sog': {
        'r2_valid_metrics': sog_dir / 'run/checkpoints/R2_E+F/plot/valid/metrics.yaml',
        'maef_valid_metrics': sog_dir / 'run/checkpoints/MAE_F/plot/valid/metrics.yaml',
        'config': sog_dir / 'run/checkpoints/R2_E+F/config.yaml',
        'log': sog_dir / 'log_92.out',
    },
    'coulomb': {
        'r2_valid_metrics': coulomb_dir / 'run/checkpoints/R2_E+F/plot/valid/metrics.yaml',
        'maef_valid_metrics': coulomb_dir / 'run/checkpoints/MAE_F/plot/valid/metrics.yaml',
        'config': coulomb_dir / 'run/checkpoints/R2_E+F/config.yaml',
    },
}

for family, sub in paths.items():
    for k, p in sub.items():
        if not p.exists():
            raise FileNotFoundError(f'[{family}] missing {k}: {p}')

print('All required files exist.')

All required files exist.


In [2]:
def read_yaml(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

def flatten_metrics(d: dict, prefix: str):
    return {
        f'{prefix}_energy_mae': float(d['energy']['mae']),
        f'{prefix}_energy_rmse': float(d['energy']['rmse']),
        f'{prefix}_forces_mae': float(d['forces']['mae']),
        f'{prefix}_forces_rmse': float(d['forces']['rmse']),
    }

rows = []
for family in ['sog', 'coulomb']:
    r2m = read_yaml(paths[family]['r2_valid_metrics'])
    maefm = read_yaml(paths[family]['maef_valid_metrics'])
    cfg = read_yaml(paths[family]['config'])

    row = {
        'kernel': family,
        **flatten_metrics(r2m, 'r2ckpt'),
        **flatten_metrics(maefm, 'maefckpt'),
        'num_parameters': int(cfg['num_parameters']),
        'n_train': int(cfg['n_train']),
        'n_valid': int(cfg['n_valid']),
        'lr_kernel_type': cfg.get('lr_kernel_type', 'coulomb(default)'),
        'sog_num_gaussians': cfg.get('sog_num_gaussians', None),
        'sog_init_mode': cfg.get('sog_init_mode', None),
    }

    for p in ['r2ckpt', 'maefckpt']:
        row[f'{p}_energy_mse'] = row[f'{p}_energy_rmse'] ** 2
        row[f'{p}_forces_mse'] = row[f'{p}_forces_rmse'] ** 2

    rows.append(row)

df = pd.DataFrame(rows).set_index('kernel')
display(df)


,r2ckpt_energy_mae,r2ckpt_energy_rmse,r2ckpt_forces_mae,r2ckpt_forces_rmse,maefckpt_energy_mae,maefckpt_energy_rmse,maefckpt_forces_mae,maefckpt_forces_rmse,num_parameters,n_train,n_valid,lr_kernel_type,sog_num_gaussians,sog_init_mode,r2ckpt_energy_mse,r2ckpt_forces_mse,maefckpt_energy_mse,maefckpt_forces_mse
kernel,,,,,,,,,,,,,,,,,,
sog,22.247226,49.723737,47.124772,80.763539,26.640575,53.539229,34.713209,71.801649,1021222,2000,500,sog,12.0,dimer_cc,2472.449979,6522.749259,2866.449028,5155.476824
coulomb,18.370082,51.555950,37.714356,74.661112,18.259697,52.473119,36.183198,73.039339,1021198,2000,500,coulomb(default),NaN,None,2658.015930,5574.281638,2753.428228,5334.745016


In [3]:
def rel_improve(sog, ref):
    # smaller is better => positive value means improved
    return (ref - sog) / ref * 100.0

comparison = pd.DataFrame({
    'metric': [
        'R2ckpt Energy RMSE',
        'R2ckpt Energy MSE',
        'R2ckpt Forces RMSE',
        'R2ckpt Forces MSE',
        'MAE_F ckpt Energy RMSE',
        'MAE_F ckpt Energy MSE',
        'MAE_F ckpt Forces RMSE',
        'MAE_F ckpt Forces MSE',
    ],
    'SOG': [
        df.loc['sog', 'r2ckpt_energy_rmse'],
        df.loc['sog', 'r2ckpt_energy_mse'],
        df.loc['sog', 'r2ckpt_forces_rmse'],
        df.loc['sog', 'r2ckpt_forces_mse'],
        df.loc['sog', 'maefckpt_energy_rmse'],
        df.loc['sog', 'maefckpt_energy_mse'],
        df.loc['sog', 'maefckpt_forces_rmse'],
        df.loc['sog', 'maefckpt_forces_mse'],
    ],
    'Coulomb': [
        df.loc['coulomb', 'r2ckpt_energy_rmse'],
        df.loc['coulomb', 'r2ckpt_energy_mse'],
        df.loc['coulomb', 'r2ckpt_forces_rmse'],
        df.loc['coulomb', 'r2ckpt_forces_mse'],
        df.loc['coulomb', 'maefckpt_energy_rmse'],
        df.loc['coulomb', 'maefckpt_energy_mse'],
        df.loc['coulomb', 'maefckpt_forces_rmse'],
        df.loc['coulomb', 'maefckpt_forces_mse'],
    ],
})

comparison['SOG vs Coulomb improvement (%)'] = comparison.apply(
    lambda r: rel_improve(r['SOG'], r['Coulomb']), axis=1
)
display(comparison)

delta_params = int(df.loc['sog', 'num_parameters'] - df.loc['coulomb', 'num_parameters'])
print(f'Parameter count delta (SOG - Coulomb): {delta_params}')
print('Expected extra SOG params for 12 Gaussians: 24 (12 amplitudes + 12 log_widths)')


,metric,SOG,Coulomb,SOG vs Coulomb improvement (%)
0,R2ckpt Energy RMSE,49.723737,51.555950,3.553834
1,R2ckpt Energy MSE,2472.449979,2658.015930,6.981371
2,R2ckpt Forces RMSE,80.763539,74.661112,-8.173502
3,R2ckpt Forces MSE,6522.749259,5574.281638,-17.015065
4,MAE_F ckpt Energy RMSE,53.539229,52.473119,-2.031726
5,MAE_F ckpt Energy MSE,2866.449028,2753.428228,-4.104730
6,MAE_F ckpt Forces RMSE,71.801649,73.039339,1.694552
7,MAE_F ckpt Forces MSE,5155.476824,5334.745016,3.360389


Parameter count delta (SOG - Coulomb): 24
Expected extra SOG params for 12 Gaussians: 24 (12 amplitudes + 12 log_widths)


In [4]:
import re

log_text = paths['sog']['log'].read_text(encoding='utf-8', errors='ignore')

has_kernel_line = 'Long-range kernel selection' in log_text
has_sog_param_name = ('sog_amplitud' in log_text) or ('sog_amplitudes' in log_text)
has_sog_width_name = ('sog_log_widt' in log_text) or ('sog_log_widths' in log_text)

step_hits = re.findall(r'state at step:\s*(\d+)', log_text)
last_step = int(step_hits[-1]) if step_hits else None

evidence = {
    'config lr_kernel_type': df.loc['sog', 'lr_kernel_type'],
    'config sog_num_gaussians': df.loc['sog', 'sog_num_gaussians'],
    'config sog_init_mode': df.loc['sog', 'sog_init_mode'],
    'log has kernel selection line': has_kernel_line,
    'log has sog_amplitudes symbol': has_sog_param_name,
    'log has sog_log_widths symbol': has_sog_width_name,
    'last reported step': last_step,
    'extra param count (SOG-Coulomb)': int(df.loc['sog', 'num_parameters'] - df.loc['coulomb', 'num_parameters']),
}

display(pd.DataFrame([evidence]))

print('\n结论建议：')
print('1) SOG 参数已被纳入模型并参与优化流程：配置、日志符号、参数总量差值(=24)一致。')
print('2) 本次两组结果下，SOG 并未在能量/力 RMSE 与 MSE 上稳定优于 Coulomb（需按上表逐项判断）。')
print('3) 若要更公平比较，建议固定随机种子并做多次重复实验，再报告均值±方差。')

,config lr_kernel_type,config sog_num_gaussians,config sog_init_mode,log has kernel selection line,log has sog_amplitudes symbol,log has sog_log_widths symbol,last reported step,extra param count (SOG-Coulomb)
0,sog,12.0,dimer_cc,True,True,True,3996990,24



结论建议：
1) SOG 参数已被纳入模型并参与优化流程：配置、日志符号、参数总量差值(=24)一致。
2) 本次两组结果下，SOG 并未在能量/力 RMSE 与 MSE 上稳定优于 Coulomb（需按上表逐项判断）。
3) 若要更公平比较，建议固定随机种子并做多次重复实验，再报告均值±方差。


## 用 best model 重新跑验证集（自动检测）

下面单元会优先查找 checkpoint 里的 `model.msgpack/state.msgpack`：

- 如果存在：可按 best checkpoint 反序列化模型参数并重新跑验证集，重算 RMSE/MSE；
- 如果不存在：说明当前目录只保留了摘要指标（`metrics.yaml`），无法在本目录直接复算，只能使用已保存指标做对比。

> 你当前这两个目录很可能属于后者（训练结束清理后仅保留 YAML/TXT）。

In [5]:
from pathlib import Path
import pandas as pd

families = {
    'sog': sog_dir,
    'coulomb': coulomb_dir,
}

records = []
for name, root in families.items():
    for ckpt_name in ['R2_E+F', 'MAE_F']:
        ckpt_dir = root / 'run' / 'checkpoints' / ckpt_name
        model_msgpack = ckpt_dir / 'model' / 'model.msgpack'
        state_msgpack = ckpt_dir / 'state.msgpack'
        records.append({
            'kernel': name,
            'checkpoint': ckpt_name,
            'checkpoint_dir': str(ckpt_dir),
            'has_model_msgpack': model_msgpack.exists(),
            'has_state_msgpack': state_msgpack.exists(),
        })

ckpt_files_df = pd.DataFrame(records)
display(ckpt_files_df)

all_available = bool((ckpt_files_df['has_model_msgpack'] & ckpt_files_df['has_state_msgpack']).all())
if not all_available:
    print('当前目录缺少 model.msgpack/state.msgpack，无法直接用 best model 重新前向验证集。')
    print('可行替代：使用现有 metrics.yaml 对比（上面已完成）；或从未清理的原始 run 目录恢复 msgpack 后再复算。')
else:
    print('检测到完整 msgpack checkpoint，可继续写入“重跑验证集”单元。')

,kernel,checkpoint,checkpoint_dir,has_model_msgpack,has_state_msgpack
0,sog,R2_E+F,/data/home/public/qiuqizhi/LOREM/my_experiment...,True,True
1,sog,MAE_F,/data/home/public/qiuqizhi/LOREM/my_experiment...,True,True
2,coulomb,R2_E+F,/data/home/public/qiuqizhi/LOREM/my_experiment...,True,True
3,coulomb,MAE_F,/data/home/public/qiuqizhi/LOREM/my_experiment...,True,True


检测到完整 msgpack checkpoint，可继续写入“重跑验证集”单元。


In [7]:
# 真实重跑：用 best checkpoint 的 model.msgpack 重新前向 valid 集并计算 RMSE/MSE
import sys
import numpy as np
import pandas as pd
import jax

# 确保可导入 marathon 与 lorem 本地模块
LOREM_ROOT = '/data/home/public/qiuqizhi/LOREM'
LOREM_CODE = '/data/home/public/qiuqizhi/LOREM/lorem'
if LOREM_ROOT not in sys.path:
    sys.path.insert(0, LOREM_ROOT)
if LOREM_CODE not in sys.path:
    sys.path.insert(0, LOREM_CODE)

from marathon.io import read_yaml, read_msgpack, from_dict
from marathon.data import datasets as marathon_datasets
from marathon.extra.hermes import DataSource, FilterEmpty
from marathon.extra.hermes.pain import Record, RecordMetadata
from transforms import ToSample, SetUpEwald, ToFixedShapeBatch


DATASETS_ROOT = Path('/data/home/public/qiuqizhi/LOREM/datasets')
if not DATASETS_ROOT.exists():
    if marathon_datasets is None:
        raise RuntimeError(
            'DATASETS_ROOT not found and marathon.data.datasets is None. '
            'Please set DATASETS env or create /data/home/public/qiuqizhi/LOREM/datasets.'
        )
    DATASETS_ROOT = Path(marathon_datasets)


def _build_valid_batches(config: dict, species_to_weight: dict):
    data_valid = DATASETS_ROOT / 'cumulene_valid'
    num_graphs = int(config['training_pipeline']['num_graphs'])
    num_nodes = int(config['training_pipeline']['num_nodes'])
    num_edges = int(config['training_pipeline']['num_edges'])

    model_cfg = config['model']['lorem.Lorem']
    cutoff = float(model_cfg['cutoff'])

    to_sample = ToSample(cutoff=cutoff, energy=True, forces=True, stress=False)
    prepare_ewald = SetUpEwald(lr_wavelength=cutoff / 8, smearing=cutoff / 4)
    batcher = ToFixedShapeBatch(num_graphs=num_graphs, num_edges=num_edges, num_nodes=num_nodes)

    source_valid = DataSource(data_valid, species_to_weight=species_to_weight)

    def valid_iterator():
        filterer = FilterEmpty()
        for i in range(len(source_valid)):
            sample = to_sample.map(source_valid[i])
            if filterer.filter(sample):
                yield Record(data=sample, metadata=RecordMetadata(index=i, record_key=i))

    return [prepare_ewald.map(b.data) for b in batcher(valid_iterator())]


def _predict_and_collate(model, params, batches):
    pred_fn = jax.jit(lambda p, b: model.predict(p, b, stress=False))

    preds_energy, refs_energy = [], []
    preds_forces, refs_forces = [], []

    for batch in batches:
        out = pred_fn(params, batch)

        # energy
        m_e = np.asarray(batch.labels['energy_mask'])
        if np.any(m_e):
            preds_energy.append(np.asarray(out['energy'])[m_e])
            refs_energy.append(np.asarray(batch.labels['energy'])[m_e])

        # forces
        m_f = np.asarray(batch.labels['forces_mask'])
        if np.any(m_f):
            preds_forces.append(np.asarray(out['forces'])[m_f])
            refs_forces.append(np.asarray(batch.labels['forces'])[m_f])

    pred_e = np.concatenate([x.reshape(-1) for x in preds_energy], axis=0)
    ref_e = np.concatenate([x.reshape(-1) for x in refs_energy], axis=0)
    pred_f = np.concatenate([x.reshape(-1, 3) for x in preds_forces], axis=0)
    ref_f = np.concatenate([x.reshape(-1, 3) for x in refs_forces], axis=0)

    return pred_e, ref_e, pred_f, ref_f


def _calc_metrics(pred_e, ref_e, pred_f, ref_f):
    err_e = pred_e - ref_e
    err_f = pred_f - ref_f

    energy_mse = float(np.mean(err_e ** 2))
    energy_rmse = float(np.sqrt(energy_mse))
    forces_mse = float(np.mean(err_f ** 2))
    forces_rmse = float(np.sqrt(forces_mse))

    return {
        'energy_rmse_rerun': energy_rmse,
        'energy_mse_rerun': energy_mse,
        'forces_rmse_rerun': forces_rmse,
        'forces_mse_rerun': forces_mse,
        'n_energy_samples': int(pred_e.shape[0]),
        'n_force_vectors': int(pred_f.shape[0]),
    }


def rerun_from_checkpoint(exp_dir: Path, ckpt_name: str):
    ckpt_dir = exp_dir / 'run' / 'checkpoints' / ckpt_name
    model_yaml = ckpt_dir / 'model' / 'model.yaml'
    model_msgpack = ckpt_dir / 'model' / 'model.msgpack'
    baseline_yaml = ckpt_dir / 'model' / 'baseline.yaml'
    config_yaml = ckpt_dir / 'config.yaml'
    saved_metrics_yaml = ckpt_dir / 'plot' / 'valid' / 'metrics.yaml'

    for p in [model_yaml, model_msgpack, baseline_yaml, config_yaml, saved_metrics_yaml]:
        if not p.exists():
            raise FileNotFoundError(f'Missing required file: {p}')

    model_dict = read_yaml(model_yaml)
    model = from_dict(model_dict)
    params = read_msgpack(model_msgpack)
    baseline = read_yaml(baseline_yaml)
    config = read_yaml(config_yaml)
    saved = read_yaml(saved_metrics_yaml)

    batches = _build_valid_batches(config=config, species_to_weight=baseline['elemental'])
    pred_e, ref_e, pred_f, ref_f = _predict_and_collate(model, params, batches)
    rerun = _calc_metrics(pred_e, ref_e, pred_f, ref_f)

    out = {
        'checkpoint': ckpt_name,
        'saved_energy_rmse': float(saved['energy']['rmse']),
        'saved_forces_rmse': float(saved['forces']['rmse']),
        'saved_energy_mse': float(saved['energy']['rmse']) ** 2,
        'saved_forces_mse': float(saved['forces']['rmse']) ** 2,
        **rerun,
    }
    out['delta_energy_rmse(rerun-saved)'] = out['energy_rmse_rerun'] - out['saved_energy_rmse']
    out['delta_forces_rmse(rerun-saved)'] = out['forces_rmse_rerun'] - out['saved_forces_rmse']
    return out


rows = []
for kernel_name, exp_dir in [('sog', sog_dir), ('coulomb', coulomb_dir)]:
    for ckpt_name in ['R2_E+F', 'MAE_F']:
        r = rerun_from_checkpoint(exp_dir, ckpt_name)
        r['kernel'] = kernel_name
        rows.append(r)

rerun_df = pd.DataFrame(rows)
display(rerun_df[['kernel','checkpoint',
                  'saved_energy_rmse','energy_rmse_rerun','delta_energy_rmse(rerun-saved)',
                  'saved_forces_rmse','forces_rmse_rerun','delta_forces_rmse(rerun-saved)',
                  'energy_mse_rerun','forces_mse_rerun','n_energy_samples','n_force_vectors']])

2026-04-27 19:29:56.502428: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-04-27 19:29:56.502461: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-04-27 19:29:56.502474: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-04-27 19:29:56.502487: W external/xla/xla/service/gpu/au

,kernel,checkpoint,saved_energy_rmse,energy_rmse_rerun,delta_energy_rmse(rerun-saved),saved_forces_rmse,forces_rmse_rerun,delta_forces_rmse(rerun-saved),energy_mse_rerun,forces_mse_rerun,n_energy_samples,n_force_vectors
0,sog,R2_E+F,49.723737,0.049898,-49.673839,80.763539,0.080791,-80.682748,0.002490,0.006527,500,6500
1,sog,MAE_F,53.539229,0.053647,-53.485582,71.801649,0.071789,-71.729860,0.002878,0.005154,500,6500
2,coulomb,R2_E+F,51.555950,0.052050,-51.503900,74.661112,0.074684,-74.586428,0.002709,0.005578,500,6500
3,coulomb,MAE_F,52.473119,0.053004,-52.420115,73.039339,0.073055,-72.966284,0.002809,0.005337,500,6500


In [ ]:
# 只看“重跑后的”SOG vs Coulomb 对比（按 checkpoint 分组）

def improve_pct(sog, ref):
    return (ref - sog) / ref * 100.0

cmp_rows = []
for ckpt_name in ['R2_E+F', 'MAE_F']:
    sog_r = rerun_df[(rerun_df.kernel == 'sog') & (rerun_df.checkpoint == ckpt_name)].iloc[0]
    cou_r = rerun_df[(rerun_df.kernel == 'coulomb') & (rerun_df.checkpoint == ckpt_name)].iloc[0]

    cmp_rows.append({
        'checkpoint': ckpt_name,
        'energy_rmse_sog': sog_r['energy_rmse_rerun'],
        'energy_rmse_coulomb': cou_r['energy_rmse_rerun'],
        'energy_mse_sog': sog_r['energy_mse_rerun'],
        'energy_mse_coulomb': cou_r['energy_mse_rerun'],
        'forces_rmse_sog': sog_r['forces_rmse_rerun'],
        'forces_rmse_coulomb': cou_r['forces_rmse_rerun'],
        'forces_mse_sog': sog_r['forces_mse_rerun'],
        'forces_mse_coulomb': cou_r['forces_mse_rerun'],
        'energy_rmse_improve_pct': improve_pct(sog_r['energy_rmse_rerun'], cou_r['energy_rmse_rerun']),
        'energy_mse_improve_pct': improve_pct(sog_r['energy_mse_rerun'], cou_r['energy_mse_rerun']),
        'forces_rmse_improve_pct': improve_pct(sog_r['forces_rmse_rerun'], cou_r['forces_rmse_rerun']),
        'forces_mse_improve_pct': improve_pct(sog_r['forces_mse_rerun'], cou_r['forces_mse_rerun']),
    })

rerun_cmp_df = pd.DataFrame(cmp_rows)
display(rerun_cmp_df)